# Novel Image and Spectrum Discovery in Autonomous PVSK IV Experiment

**Author:** Yongtao Liu  
**Date:** January, 2026         

---

## Overview

This notebook implements a **novelty-driven autonomous experiment** on a perovskite (PVSK) sample using an Asylum Cypher AFM. The workflow combines:

1. **Structural imaging** — A survey AFM image is loaded and tessellated into local image patches.
2. **Image novelty scoring** — Structural dissimilarity (SSIM-based) quantifies how "unique" each unmeasured patch looks relative to already-measured ones.
3. **Spectrum novelty scoring** — Isolation Forest anomaly detection flags IV curves that are statistically unusual within the measured set.
4. **Deep Kernel Learning (DKL) surrogate** — A DKL Gaussian Process is trained on patch features vs. spectrum novelty, then used to predict novelty at unmeasured locations.
5. **Novelty-constrained acquisition** — Upper Confidence Bound (UCB) acquisition values are multiplied by image novelty scores, so the agent preferentially visits structurally diverse, high-novelty locations.
6. **Closed-loop measurement** — The Cypher AFM tip is autonomously repositioned and IV curves are acquired iteratively.

---

## DKL Backend Options

Two DKL implementations are supported and can be used interchangeably in this workflow:

| Backend | Package | Install | Import |
|---|---|---|---|
| **GPax `viDKL`** *(used by default)* | [`gpax`](https://github.com/ziatdinovmax/gpax) | `pip install gpax==0.1.8` | `import gpax` |
| **ENGAGEGP `fit_dkgp`** *(alternative)* | [`engage-gp`](https://github.com/yongtaoliu/ENGAGEGP) | `pip install engage-gp` | `from engagegp import fit_dkgp, predict_dkgpr` |

### When to prefer ENGAGEGP

ENGAGEGP (`engage-gp`) is a PyTorch-based DKL library developed alongside this experiment framework. It offers several advantages over GPax for certain use cases:

- **Richer feature extractors** — choose from `fc`, `fcbn`, `resnet`, `attention`, `direct_attention`, or `attention_weighted` architectures via a simple factory function.
- **Explainability** — attention-based extractors expose per-feature importance maps and head×head attention matrices, making it easier to understand *which* patch regions drive predictions.
- **Preference learning** — supports pairwise GP (`fit_dkgppw`) if you want to steer acquisition via human comparisons rather than scalar labels.
- **Sample weighting** — learnable per-sample confidence weights handle heteroscedastic or unreliable measurements.
- **Broader acquisition functions** — EI, UCB, PI, Thompson sampling, and constrained EI are all built in.

### ENGAGEGP usage example (drop-in replacement for GPax `viDKL`)

```python
from engagegp import fit_dkgp, predict_dkgpr, upper_confidence_bound

# Fit DKL surrogate (PyTorch backend)
mll, gp, dkl, losses = fit_dkgp(
    X_measured, y_measured,
    feature_dim=16,
    extractor_type='fcbn',          # or 'resnet', 'attention', etc.
    num_epochs=500,
    lr_features=1e-4,
    lr_gp=1e-2,
)

# Predict at unmeasured locations
y_pred, y_std = predict_dkgpr(dkl, X_unmeasured, return_std=True)
y_var = y_std ** 2

# UCB acquisition (replaces gpax.acquisition.UCB)
obj = upper_confidence_bound(dkl, X_unmeasured, beta=2.0)
```

> **Hardware dependency:** Cells that call `Cypher.PyCypher()` require a live connection to the Asylum Cypher or VERO AFM controller and are not executable offline.


---
## 1. Environment Setup

### 1.1 Installation

Uncomment the lines below on first run to install required packages. Pin the versions shown for reproducibility with GPax and JAX.


In [ ]:
# Uncomment to install on first run (pinned versions for JAX/GPax compatibility)
# !pip install -q gpax==0.1.8
# !pip install -q atomai
# !pip install sidpy pyNSID bglib SciFiReaders
# !pip install --upgrade numpy==1.26.4
# !pip install --upgrade jax==0.4.28 jaxlib==0.4.28 numpyro==0.13.2
# !pip install aecroscopywave

### 1.2 Imports

All imports are consolidated here. `win32com.client` is required on Windows for COM-based instrument control. `gpax.utils.enable_x64()` enables 64-bit floating-point precision in JAX, which improves numerical stability for GP inference.


In [ ]:
import os
import re
import time

import gpax
import scipy
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from warnings import filterwarnings

import SciFiReaders

from skimage.metrics import structural_similarity as ssim
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from scipy.signal import savgol_filter
from atomai.utils import get_coord_grid, extract_subimages

# Cypher AFM Python interface (instrument-specific; not available offline)
from aecroscopywave.interfaces import Cypher

filterwarnings("ignore")
gpax.utils.enable_x64()    # Enable 64-bit precision in JAX for stable GP inference


---
## 2. Configuration

Centralise all user-facing parameters here so they are easy to adjust without digging into workflow cells.


In [ ]:
# ── Data paths ──────────────────────────────────────────────────────────────
DATA_DIR       = r"Your Data Directory"
STRUCT_IMG_FILE = "strucimg0000.ibw"    # Survey AFM image (IBW format)
IV_BASE_NAME   = "iv"                  # Prefix for IV curve files

# ── Image tessellation ───────────────────────────────────────────────────────
GRID_STEP      = 2       # Step size (pixels) between patch centres on the grid
WINDOW_SIZE    = 16      # Patch side length in pixels (patches are WINDOW_SIZE × WINDOW_SIZE)

# ── Initial random sampling (Latin Hypercube) ────────────────────────────────
N_INIT         = 10      # Number of initial IV measurements before active learning
RANDOM_SEED    = 1       # NumPy seed for reproducibility
LHS_SEED       = 10      # Latin Hypercube sampler seed

# ── Active learning loop ─────────────────────────────────────────────────────
N_ITER         = 200     # Total active-learning iterations
DKL_STEPS      = 2000    # Variational inference optimisation steps per iteration
DKL_STEP_SIZE  = 0.0001  # Learning rate for DKL fitting
DKL_LATENT_DIM = 2       # Latent space dimensionality for DKL
DKL_KERNEL     = "Matern"

# ── Spectrum novelty (Isolation Forest) ─────────────────────────────────────
IF_CONTAMINATION = 0.1   # Expected fraction of anomalous spectra

# ── Signal pre-processing ────────────────────────────────────────────────────
DOWNSAMPLE_FACTOR = 20   # IV curve downsampling factor (average pooling)
SMOOTH_WINDOW     = 11   # Savitzky-Golay filter window length
POLY_ORDER        = 1    # Savitzky-Golay polynomial order

# ── Instrument timing ────────────────────────────────────────────────────────
TIP_MOVE_DELAY    = 2    # Seconds to wait after moving tip before measurement
POLL_INTERVAL     = 1    # Seconds between file-existence checks

---
## 3. Utility Functions

### 3.1 Normalisation


In [ ]:
def norm_(x: np.ndarray) -> np.ndarray:
    """Min-max normalise an array to [0, 1]."""
    return (x - x.min()) / np.ptp(x)

### 3.2 IV Curve Pre-processing

Raw IV curves from the Cypher are sampled at high frequency. `downsample_and_smooth` reduces the data volume by:
1. Trimming the signal symmetrically so its length is divisible by `downsample_factor`.
2. Averaging contiguous blocks of `downsample_factor` samples (mean pooling).
3. Applying a Savitzky-Golay filter to suppress high-frequency noise while preserving peak shapes.


In [ ]:
def downsample_and_smooth(
    y: np.ndarray,
    downsample_factor: int = DOWNSAMPLE_FACTOR,
    smooth_window: int = SMOOTH_WINDOW,
    polyorder: int = POLY_ORDER,
) -> np.ndarray:
    """Downsample and smooth a 1-D IV spectrum.

    Parameters
    ----------
    y : array_like
        Raw current signal.
    downsample_factor : int
        Number of raw samples averaged into one output sample.
    smooth_window : int
        Savitzky-Golay filter window length (must be odd and > polyorder).
    polyorder : int
        Polynomial order for Savitzky-Golay filter.

    Returns
    -------
    np.ndarray
        Smoothed, downsampled spectrum.
    """
    y = np.asarray(y)
    n = (len(y) // downsample_factor) * downsample_factor
    start_n = (len(y) - n) // 2
    end_n = len(y) - n - start_n
    y_ds = y[start_n : len(y) - end_n].reshape(-1, downsample_factor).mean(axis=1)
    return savgol_filter(y_ds, smooth_window, polyorder)


### 3.3 File-name Generation

The Cypher or VERO saves each IBW file with a zero-padded 4-digit index appended to a base name (e.g. `iv0003.ibw`). `get_next_filename` scans the target directory to find the next unused index.


In [ ]:
def get_next_filename(base_name: str, directory: str, filetype: str) -> str:
    """Return the next sequential filename with a 4-digit zero-padded suffix.

    Parameters
    ----------
    base_name : str
        File base name (e.g. "iv").
    directory : str
        Directory to scan for existing files.
    filetype : str
        File extension including dot (e.g. ".ibw").

    Returns
    -------
    str
        Next available filename (e.g. "iv0004.ibw").
    """
    pattern = re.compile(rf"^{re.escape(base_name)}(\d{{4}})\..+$")
    max_index = -1
    for filename in os.listdir(directory):
        match = pattern.match(filename)
        if match:
            idx = int(match.group(1))
            if idx > max_index:
                max_index = idx
    return f"{base_name}{max_index + 1:04d}{filetype}"


### 3.4 IBW File Reader

Wraps `SciFiReaders.IgorIBWReader` to load Igor Binary Wave (`.ibw`) files produced by the Asylum Cypher. Each file contains multiple data channels; channel 0 is current and channel 1 is bias voltage.


In [ ]:
def read_ibw(ibw_file: str):
    """Read an Igor IBW file and return its data channels.

    Parameters
    ----------
    ibw_file : str
        Path to the .ibw file.

    Returns
    -------
    list
        List of channel arrays. Channel 0 = current; channel 1 = bias.
    """
    reader = SciFiReaders.IgorIBWReader(ibw_file)
    return reader.read()


---
## 4. Novelty Scoring

Two complementary novelty metrics are used — one for IV spectra and one for image patches. Together they steer the agent toward locations that are both structurally unusual *and* electrically anomalous.

### 4.1 Spectrum Novelty — Isolation Forest

`spectra_novelty` fits an Isolation Forest on the current set of measured IV curves and returns an anomaly score for each. Isolation Forest assigns higher scores to spectra that require fewer splits to isolate — i.e. spectra that are statistically distant from the bulk of the dataset.

**Key property:** The scores are recomputed from scratch at each iteration as new spectra are added, so the definition of "novel" evolves with the measured dataset.


In [ ]:
def spectra_novelty(dataset: np.ndarray) -> np.ndarray:
    """Compute novelty scores for a set of IV spectra via Isolation Forest.

    Higher score = more anomalous / novel spectrum.

    Parameters
    ----------
    dataset : np.ndarray, shape (n_spectra, n_features)
        Normalised IV curves, one row per spectrum.

    Returns
    -------
    np.ndarray, shape (n_spectra,)
        Novelty scores (higher = more novel).
    """
    clf = IsolationForest(contamination=IF_CONTAMINATION, random_state=RANDOM_SEED)
    clf.fit(dataset)
    # decision_function returns negative scores for anomalies; negate so higher = more novel
    return -clf.decision_function(dataset)


### 4.2 Image Novelty — SSIM-based `NoveltyCalculator`

`NoveltyCalculator` quantifies how structurally *different* each unmeasured image patch is from all patches that have already been measured. It uses the **Structural Similarity Index (SSIM)** as the similarity kernel.

**Efficient incremental updates:** Rather than recomputing all pairwise SSIMs from scratch after each acquisition, only the column corresponding to the newly measured patch is appended to the cached SSIM matrix. This reduces per-iteration complexity from O(N·M) to O(N).

**Novelty score:** For each unmeasured patch i, the novelty score is `1 − max_j SSIM(patch_i, ref_j)`, so patches that look very different from *all* measured patches score close to 1.


In [ ]:
class NoveltyCalculator:
    """Incremental SSIM-based image novelty calculator for active learning.

    Maintains a live SSIM matrix [num_target × num_ref] and performs O(N)
    updates when a patch moves from the target (unmeasured) set to the
    reference (measured) set.

    Parameters
    ----------
    reference_images : np.ndarray, shape (num_ref, H, W)
        Initially measured image patches.
    target_images : np.ndarray, shape (num_target, H, W)
        All unmeasured image patches.
    """

    def __init__(self, reference_images: np.ndarray, target_images: np.ndarray):
        self.reference_images = reference_images.copy()
        self.target_images = target_images.copy()
        self.num_ref = len(reference_images)
        self.num_target = len(target_images)
        self.ssim_matrix = None  # Filled by compute_initial_ssim()
        print(f"Initialized with {self.num_ref} reference and {self.num_target} target images")

    def compute_initial_ssim(self, show_progress: bool = True):
        """Compute the full SSIM matrix once at initialisation.

        Cost: O(num_target × num_ref). Should only be called once.

        Parameters
        ----------
        show_progress : bool
            Display a tqdm progress bar.
        """
        self.ssim_matrix = np.zeros((self.num_target, self.num_ref))
        iterator = (
            tqdm(range(self.num_target), desc="Computing initial SSIM matrix")
            if show_progress else range(self.num_target)
        )
        for i in iterator:
            for j in range(self.num_ref):
                self.ssim_matrix[i, j] = ssim(
                    self.target_images[i],
                    self.reference_images[j],
                    data_range=1,
                )

    def get_novelty_scores(self) -> np.ndarray:
        """Return novelty scores for all current target images.

        Score = 1 − max_j SSIM(target_i, ref_j).
        A score near 1 means the patch looks unlike anything measured so far.

        Returns
        -------
        np.ndarray, shape (num_target,)
        """
        if self.ssim_matrix is None or self.ssim_matrix.shape[1] == 0:
            return np.ones(self.num_target)
        max_ssims = np.max(self.ssim_matrix[: self.num_target, : self.num_ref], axis=1)
        return 1 - max_ssims

    def update_novelty_matrix(self, target_idx: int, show_progress: bool = False):
        """Move a target patch to the reference set (incremental O(N) update).

        Only computes SSIM between the remaining target patches and the
        newly promoted reference image — no full recomputation.

        Parameters
        ----------
        target_idx : int
            Index of the target patch to promote to the reference set.
        show_progress : bool
            Display a tqdm progress bar for the new SSIM column.
        """
        moved_image = self.target_images[target_idx : target_idx + 1]
        self.reference_images = np.concatenate([self.reference_images, moved_image], axis=0)

        # Compute SSIM between all remaining targets and the newly added reference
        new_column = np.zeros((self.num_target, 1))
        iterator = (
            tqdm(range(self.num_target), desc="Updating SSIM column", disable=not show_progress)
            if show_progress else range(self.num_target)
        )
        for i in iterator:
            new_column[i, 0] = ssim(
                self.target_images[i],
                self.target_images[target_idx],
                data_range=1,
            )

        self.ssim_matrix = np.concatenate([self.ssim_matrix, new_column], axis=1)

        # Remove the promoted patch from the target set
        self.target_images = np.delete(self.target_images, target_idx, axis=0)
        self.ssim_matrix = np.delete(self.ssim_matrix, target_idx, axis=0)
        self.num_ref += 1
        self.num_target -= 1

---
## 5. Load and Prepare Structural Image

The survey AFM image is loaded from an IBW file, transposed to match the display orientation, normalised, and tessellated into overlapping `WINDOW_SIZE × WINDOW_SIZE` patches on a regular grid. Each patch will serve as the feature vector for one candidate measurement location.


In [ ]:
os.chdir(DATA_DIR)

# Load and orient the structural image
struc_img = np.asarray(read_ibw(STRUCT_IMG_FILE)[0]).T   # Transpose to (rows, cols)

plt.figure(figsize=(5, 5))
plt.imshow(struc_img, cmap="gray", origin="lower")
plt.title("Raw Structural AFM Image")
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"Image shape: {struc_img.shape}")

### 5.1 Image Tessellation

The normalised image is sampled on a uniform grid (step = `GRID_STEP` px). At each grid point a `WINDOW_SIZE × WINDOW_SIZE` patch is extracted using `atomai.utils.extract_subimages`. The collection of all patches forms the feature matrix **X** with shape `(n_patches, WINDOW_SIZE²)`.


In [ ]:
img = norm_(struc_img)

# Generate grid coordinates
coordinates = get_coord_grid(img, step=GRID_STEP, return_dict=False)

# Extract local patches at every grid point
features_all, coords, _ = extract_subimages(img, coordinates, WINDOW_SIZE)
features_all = features_all[:, :, :, 0]            # Drop trailing singleton channel dim
coords = np.array(coords, dtype=int)

print(f"Total patches: {features_all.shape[0]}")
print(f"Patch shape:   {features_all.shape[1:]}")
print(f"Coords shape:  {coords.shape}")

# Normalise patches to [0, 1]
features_all = norm_(features_all)

# Visualise a representative patch
k = 100
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3), dpi=100)
ax1.imshow(img, origin="lower", cmap="gray")
ax1.scatter(coords[k, 1], coords[k, 0], marker="x", s=60, c="red", label=f"Patch #{k}")
ax1.set_title("Full Image with Sample Patch Location")
ax1.legend(fontsize=8)
ax1.axis("off")
ax2.imshow(features_all[k], origin="lower", cmap="gray")
ax2.set_title(f"Extracted Patch #{k} ({WINDOW_SIZE}×{WINDOW_SIZE} px)")
ax2.axis("off")
plt.tight_layout()
plt.show()

# Flatten patches into feature vectors
indices_all = coords
X = features_all.reshape(len(features_all), WINDOW_SIZE * WINDOW_SIZE)
print(f"Feature matrix X shape: {X.shape}")


---
## 6. Initial Latin Hypercube Sampling

Before active learning begins, `N_INIT` measurement locations are selected using **Latin Hypercube Sampling (LHS)** to achieve near-uniform spatial coverage. This provides a diverse initial training set for the DKL surrogate.


In [ ]:
np.random.seed(RANDOM_SEED)
sampler = scipy.stats.qmc.LatinHypercube(d=1, seed=LHS_SEED)
train_idx = np.round(
    (len(features_all) - 1) * sampler.random(n=N_INIT)
).astype(int).ravel()

train_x       = X[train_idx]
train_indices = indices_all[train_idx]

print(f"Initial training set: {N_INIT} points")
print(f"train_x shape:        {train_x.shape}")
print(f"Train indices:
{train_indices}")

# Visualise initial sampling locations on the structural image
plt.figure(figsize=(5, 5))
plt.imshow(struc_img, origin="lower", cmap="gray")
sc = plt.scatter(
    train_indices[:, 1], train_indices[:, 0],
    c=np.arange(N_INIT), cmap="jet", marker="s", s=60, alpha=0.9,
)
plt.colorbar(sc, label="Acquisition order")
plt.title(f"Initial {N_INIT} LHS Measurement Locations")
plt.axis("off")
plt.tight_layout()
plt.show()


---
## 7. Instrument Connection

Connect to the Cypher AFM controller via the `PyCypher` Python interface. This cell requires a live instrument; skip or mock it for offline analysis.


In [ ]:
# ⚠️  Requires live Cypher AFM connection
pycy = Cypher.PyCypher()
gmv  = pycy.get_MasterVariables()     # Cache master variable list for later use
pycy.show_tip()                        # Confirm current tip position in software


---
## 8. Initial IV Measurements

The AFM tip is moved to each LHS-selected location and an IV curve is acquired. The instrument saves each curve as a `.ibw` file; `get_next_filename` resolves the correct filename by scanning the directory for the most recently written file.

After collection, `spectra_novelty` scores every curve via Isolation Forest, and the scores (normalised to [0,1]) are used as the initial target values for the DKL surrogate.


In [ ]:
directory = DATA_DIR + "\\"

ivcurves = []
for index in train_indices:
    pycy.move_tip([index[1], index[0]])
    time.sleep(TIP_MOVE_DELAY)
    pycy.DoIV()

    next_filename = get_next_filename(IV_BASE_NAME, directory, ".ibw")
    loop_path = directory + next_filename
    print(f"Waiting for: {next_filename}")
    while not os.path.exists(loop_path):
        time.sleep(POLL_INTERVAL)

    current  = read_ibw(loop_path)[0]
    iv_curve = np.array(downsample_and_smooth(current))
    time.sleep(POLL_INTERVAL)
    ivcurves.append(iv_curve)

print(f"\nCollected {len(ivcurves)} initial IV curves.")
print(f"IV curve length after downsampling: {ivcurves[0].shape[0]} points")


### 8.1 Initial Spectrum Novelty Scores

Compute and visualise the Isolation Forest novelty scores for the initial set of measured IV curves. Locations with higher scores correspond to more anomalous local electronic behaviour.


In [ ]:
y_measured_unnor = spectra_novelty(dataset=norm_(np.asarray(ivcurves)))

# Map novelty scores back onto the image
plt.figure(figsize=(5, 5))
plt.imshow(struc_img, origin="lower", cmap="gray")
sc = plt.scatter(
    train_indices[:, 1], train_indices[:, 0],
    c=y_measured_unnor, cmap="jet", marker="s", s=80, alpha=0.9,
)
plt.colorbar(sc, label="Spectrum Novelty Score")
plt.title("Initial Spectrum Novelty Scores (Isolation Forest)")
plt.axis("off")
plt.tight_layout()
plt.show()


---
## 9. Active Learning Setup

Split the full patch/coordinate arrays into measured and unmeasured subsets, then initialise the `NoveltyCalculator` with the initial measured patches to pre-compute the SSIM matrix.


In [ ]:
# Split measured / unmeasured
X_measured       = train_x
y_measured       = norm_(y_measured_unnor)
indices_measured = train_indices

X_unmeasured       = np.delete(X, train_idx, axis=0)
indices_unmeasured = np.delete(indices_all, train_idx, axis=0)

print(f"Measured:   X={X_measured.shape}, y={y_measured.shape}, idx={indices_measured.shape}")
print(f"Unmeasured: X={X_unmeasured.shape}, idx={indices_unmeasured.shape}")

# Initialise image novelty calculator
print("\nBuilding initial SSIM matrix ...")
X_measured_novelty   = X_measured.reshape(-1, WINDOW_SIZE, WINDOW_SIZE)
X_unmeasured_novelty = X_unmeasured.reshape(-1, WINDOW_SIZE, WINDOW_SIZE)

noveltyImgs = NoveltyCalculator(X_measured_novelty, X_unmeasured_novelty)
noveltyImgs.compute_initial_ssim()
print("SSIM matrix ready.")


---
## 10. Visualisation Helper

`plot_prediction` produces a three-panel summary at each active learning step:
- **Left:** Structural image overlaid with measured-point novelty values.
- **Centre:** DKL posterior mean prediction over all unmeasured locations.
- **Right:** DKL posterior uncertainty (variance), highlighting where the surrogate is least confident.


In [ ]:
def plot_prediction(indices_measured, y_measured, indices_unmeasured, y_pred, y_var):
    """Three-panel DKL prediction summary plot."""
    shrink = 0.7
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

    ax1.imshow(struc_img, cmap="gray", origin="lower")
    sc1 = ax1.scatter(indices_measured[:, 1], indices_measured[:, 0],
                      c=y_measured, cmap="jet", alpha=1)
    plt.colorbar(sc1, ax=ax1, shrink=shrink)
    ax1.set_title("Measured — Spectrum Novelty")
    ax1.axis("off")

    sc2 = ax2.scatter(indices_unmeasured[:, 1], indices_unmeasured[:, 0],
                      c=y_pred, cmap="viridis")
    plt.colorbar(sc2, ax=ax2, shrink=shrink)
    ax2.set_title("DKL Posterior Mean")
    ax2.set_aspect("equal")
    ax2.axis("off")

    sc3 = ax3.scatter(indices_unmeasured[:, 1], indices_unmeasured[:, 0],
                      c=y_var, cmap="plasma")
    plt.colorbar(sc3, ax=ax3, shrink=shrink)
    ax3.set_title("DKL Posterior Uncertainty")
    ax3.set_aspect("equal")
    ax3.axis("off")

    plt.tight_layout()
    plt.show()


---
## 11. Novelty-Driven Active Learning Loop

The core experiment loop runs for `N_ITER` steps. At each step:

1. **Fit DKL** — A variational Deep Kernel Learning GP (`gpax.viDKL`) is trained on the current set of `(X_measured, y_measured)` pairs. The deep kernel maps high-dimensional image patches into a 2-D latent space before applying a Matérn covariance function.
2. **UCB acquisition** — The Upper Confidence Bound scores each unmeasured location, balancing the DKL posterior mean (exploitation) against its uncertainty (exploration).
3. **Image novelty modulation** — UCB scores are element-wise multiplied by the SSIM-based image novelty scores. This penalises locations whose patches look similar to already-measured ones, even if the GP thinks they are worth visiting.
4. **Select next point** — The location with the highest modulated acquisition score is chosen.
5. **Acquire IV curve** — The AFM tip is moved to that location and an IV curve is recorded.
6. **Update bookkeeping** — The new curve is added to `ivcurves`; novelty scores are recomputed; measured/unmeasured arrays and the SSIM matrix are updated.


In [ ]:
data_dim = X_measured.shape[-1]
key1, key2 = gpax.utils.get_keys()

for i in range(N_ITER):
    print(f"\n── Iteration {i + 1}/{N_ITER} ──────────────────────────")

    # 1. Fit DKL surrogate
    dkl = gpax.viDKL(data_dim, DKL_LATENT_DIM, kernel=DKL_KERNEL)
    dkl.fit(key1, X_measured, y_measured,
            num_steps=DKL_STEPS, step_size=DKL_STEP_SIZE)

    # 2. UCB acquisition & posterior prediction
    obj = gpax.acquisition.UCB(key2, dkl, X_unmeasured, maximize=True)
    y_pred, y_var = dkl.predict(key2, X_unmeasured)

    # 3. Image novelty modulation
    img_novelty_scores = noveltyImgs.get_novelty_scores()
    obj_novelty = obj * img_novelty_scores

    # 4. Select next measurement location
    next_point_idx = obj_novelty.argmax()
    next_coordinates = indices_unmeasured[next_point_idx]
    print(f"Next location: row={next_coordinates[0]}, col={next_coordinates[1]}")

    # Visualise surrogate state
    plot_prediction(indices_measured, y_measured, indices_unmeasured, y_pred, y_var)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
    for ax, data, title in zip(
        [ax1, ax2, ax3],
        [obj, img_novelty_scores, obj_novelty],
        ["UCB Acquisition", "Image Novelty Score", "Modulated Acquisition (UCB × Novelty)"],
    ):
        ax.scatter(indices_unmeasured[:, 1], indices_unmeasured[:, 0], c=data, s=8)
        ax.scatter(
            indices_unmeasured[next_point_idx, 1],
            indices_unmeasured[next_point_idx, 0],
            c="red", s=120, zorder=5, label="Next point",
        )
        ax.set_title(title)
        ax.set_aspect("equal")
        ax.axis("off")
    ax1.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # 5. Acquire IV curve at selected location
    pycy.move_tip([next_coordinates[1], next_coordinates[0]])
    time.sleep(TIP_MOVE_DELAY)
    pycy.DoIV()

    next_filename = get_next_filename(IV_BASE_NAME, directory, ".ibw")
    loop_path = directory + next_filename
    print(f"Waiting for: {next_filename}")
    while not os.path.exists(loop_path):
        time.sleep(POLL_INTERVAL)
    time.sleep(0.5)

    current  = read_ibw(loop_path)[0]
    iv_curve = np.array(downsample_and_smooth(current))
    ivcurves.append(iv_curve)

    # 6. Update measurements and novelty scores
    spectra_novelty_scores = spectra_novelty(dataset=norm_(np.asarray(ivcurves)))
    y_measured = norm_(spectra_novelty_scores)

    X_measured         = np.append(X_measured, X_unmeasured[[next_point_idx]], axis=0)
    X_unmeasured       = np.delete(X_unmeasured, next_point_idx, axis=0)
    indices_measured   = np.append(indices_measured, indices_unmeasured[[next_point_idx]], axis=0)
    indices_unmeasured = np.delete(indices_unmeasured, next_point_idx, axis=0)

    noveltyImgs.update_novelty_matrix(next_point_idx)

print("\n✓ Active learning loop complete.")
